# Classificação de Flores com CNNs e Vision Transformers

**Nome: Thais da Silva Ferreira**

**Disciplina:** Processamento de Imagem para IA Generativa

## Objetivo

Comparar duas abordagens pré-treinadas para classificação multiclasse de imagens do dataset Oxford Flowers 102:

1. **CNN pré-treinada:** ResNet50;
2. **Vision Transformer pré-treinado:** ViT-B/16.

A comparação considera desempenho, generalização, custo computacional, complexidade e tempo de fine-tuning.

## 1. Instalação e importação das bibliotecas

Se estiver executando no Google Colab, habilite uma GPU em **Ambiente de execução > Alterar tipo de ambiente de execução > GPU**.

In [ ]:
# Execute apenas se necessário
# !pip install -q torch torchvision scikit-learn seaborn pandas matplotlib tqdm

In [ ]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

## 2. Configuração do ambiente

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Configurações do experimento

O Oxford Flowers 102 possui 102 classes. Para reduzir o custo computacional, este notebook utiliza as divisões oficiais `train`, `val` e `test`.

Para testes rápidos, reduza `IMAGE_SIZE`, `BATCH_SIZE` e `EPOCHS`.

In [ ]:
DATA_DIR = "./data"
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 5
LR = 1e-4
NUM_CLASSES = 102

# Para uma execução mais rápida, use False.
FAST_MODE = False
FAST_TRAIN_SAMPLES = 1020
FAST_VAL_SAMPLES = 510
FAST_TEST_SAMPLES = 1020

## 4. Carregamento do dataset Oxford Flowers 102

O dataset é disponibilizado pelo `torchvision.datasets.Flowers102`.

As transformações incluem redimensionamento, normalização com estatísticas do ImageNet e aumento de dados apenas no conjunto de treino.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.Flowers102(
    root=DATA_DIR,
    split="train",
    download=True,
    transform=train_transform
)

val_dataset = datasets.Flowers102(
    root=DATA_DIR,
    split="val",
    download=True,
    transform=eval_transform
)

test_dataset = datasets.Flowers102(
    root=DATA_DIR,
    split="test",
    download=True,
    transform=eval_transform
)

print("Treino:", len(train_dataset))
print("Validação:", len(val_dataset))
print("Teste:", len(test_dataset))

### Modo rápido opcional

Útil para validar se todo o notebook está funcionando antes do treinamento completo.

In [ ]:
if FAST_MODE:
    from torch.utils.data import Subset

    train_dataset = Subset(train_dataset, range(min(FAST_TRAIN_SAMPLES, len(train_dataset))))
    val_dataset = Subset(val_dataset, range(min(FAST_VAL_SAMPLES, len(val_dataset))))
    test_dataset = Subset(test_dataset, range(min(FAST_TEST_SAMPLES, len(test_dataset))))

    print("FAST_MODE ativado.")
    print("Treino:", len(train_dataset))
    print("Validação:", len(val_dataset))
    print("Teste:", len(test_dataset))

In [ ]:
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

## 5. Exploração visual do dataset

In [ ]:
def denormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(img.cpu() * std + mean, 0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i in range(8):
    axes[i].imshow(denormalize(images[i]).permute(1, 2, 0))
    axes[i].set_title(f"Classe: {labels[i].item()}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 6. Funções de treinamento e avaliação

As métricas utilizadas serão:

- **Accuracy:** proporção total de classificações corretas;
- **F1-score macro:** média do F1 entre todas as 102 classes, dando o mesmo peso para cada classe.

O F1 macro é importante porque evita que classes com maior quantidade de exemplos dominem a avaliação.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return loss, acc, f1, np.array(all_labels), np.array(all_preds)


def fit_model(model, train_loader, val_loader, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
        "train_f1": [], "val_f1": []
    }

    best_state = None
    best_f1 = -1
    start_time = time.perf_counter()

    for epoch in range(epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer
        )

        val_loss, val_acc, val_f1, _, _ = evaluate(
            model, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        print(
            f"Época {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
            f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

    elapsed_time = time.perf_counter() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, elapsed_time

# Parte A — CNN pré-treinada

## 7. ResNet50 com Transfer Learning

A ResNet50 é uma CNN profunda baseada em conexões residuais. Essas conexões facilitam o treinamento de redes profundas e permitem reutilizar representações aprendidas no ImageNet.

Nesta primeira etapa, congelamos a maior parte da rede e treinamos principalmente a camada classificadora final.

In [ ]:
weights_resnet = models.ResNet50_Weights.DEFAULT
cnn_model = models.resnet50(weights=weights_resnet)

# Congela as camadas pré-treinadas
for param in cnn_model.parameters():
    param.requires_grad = False

# Substitui a camada final para 102 classes
in_features = cnn_model.fc.in_features
cnn_model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, NUM_CLASSES)
)

cnn_model = cnn_model.to(device)

cnn_trainable_params = sum(
    p.numel() for p in cnn_model.parameters() if p.requires_grad
)
cnn_total_params = sum(p.numel() for p in cnn_model.parameters())

print("Parâmetros totais:", f"{cnn_total_params:,}")
print("Parâmetros treináveis:", f"{cnn_trainable_params:,}")

## 8. Treinamento da CNN

In [ ]:
cnn_model, cnn_history, cnn_time = fit_model(
    cnn_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR
)

print(f"Tempo total da CNN: {cnn_time / 60:.2f} minutos")

## 9. Curvas de treinamento da CNN

In [ ]:
epochs_range = range(1, len(cnn_history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, cnn_history["train_loss"], marker="o", label="Treino")
plt.plot(epochs_range, cnn_history["val_loss"], marker="o", label="Validação")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("CNN - Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, cnn_history["train_acc"], marker="o", label="Treino")
plt.plot(epochs_range, cnn_history["val_acc"], marker="o", label="Validação")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("CNN - Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, cnn_history["train_f1"], marker="o", label="Treino")
plt.plot(epochs_range, cnn_history["val_f1"], marker="o", label="Validação")
plt.xlabel("Época")
plt.ylabel("F1-score Macro")
plt.title("CNN - F1-score Macro")
plt.legend()
plt.show()

## 10. Avaliação final da CNN

In [ ]:
criterion = nn.CrossEntropyLoss()

cnn_test_loss, cnn_test_acc, cnn_test_f1, y_true_cnn, y_pred_cnn = evaluate(
    cnn_model,
    test_loader,
    criterion
)

print("CNN - Resultados no teste")
print(f"Loss:     {cnn_test_loss:.4f}")
print(f"Accuracy: {cnn_test_acc:.4f}")
print(f"F1 Macro: {cnn_test_f1:.4f}")

# Parte B — Vision Transformer

## 11. Como funciona o Vision Transformer?

Uma imagem não é processada diretamente como uma grade inteira de pixels. O ViT divide a imagem em pequenos blocos chamados **patches**.

Para uma imagem de 224 × 224 pixels e patches de 16 × 16 pixels:

- 224 / 16 = 14 patches por dimensão;
- 14 × 14 = 196 patches no total.

Cada patch é transformado em uma representação vetorial. Em seguida, o Transformer utiliza mecanismos de **self-attention** para aprender relações entre diferentes regiões da imagem.

Isso permite que o modelo relacione partes distantes da imagem, como pétalas, folhas e padrões visuais presentes em regiões diferentes.

## 12. ViT-B/16 com Transfer Learning

Será utilizado o modelo ViT-B/16 pré-treinado. Assim como na CNN, inicialmente treinaremos apenas a cabeça classificadora.

In [ ]:
weights_vit = models.ViT_B_16_Weights.DEFAULT
vit_model = models.vit_b_16(weights=weights_vit)

# Congela o backbone
for param in vit_model.parameters():
    param.requires_grad = False

# Substitui a cabeça classificadora
in_features = vit_model.heads.head.in_features
vit_model.heads.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, NUM_CLASSES)
)

vit_model = vit_model.to(device)

vit_trainable_params = sum(
    p.numel() for p in vit_model.parameters() if p.requires_grad
)
vit_total_params = sum(p.numel() for p in vit_model.parameters())

print("Parâmetros totais:", f"{vit_total_params:,}")
print("Parâmetros treináveis:", f"{vit_trainable_params:,}")

## 13. Treinamento do Vision Transformer

In [ ]:
vit_model, vit_history, vit_time = fit_model(
    vit_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR
)

print(f"Tempo total do ViT: {vit_time / 60:.2f} minutos")

## 14. Curvas de treinamento do Vision Transformer

In [ ]:
epochs_range = range(1, len(vit_history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, vit_history["train_loss"], marker="o", label="Treino")
plt.plot(epochs_range, vit_history["val_loss"], marker="o", label="Validação")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("ViT - Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, vit_history["train_acc"], marker="o", label="Treino")
plt.plot(epochs_range, vit_history["val_acc"], marker="o", label="Validação")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("ViT - Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, vit_history["train_f1"], marker="o", label="Treino")
plt.plot(epochs_range, vit_history["val_f1"], marker="o", label="Validação")
plt.xlabel("Época")
plt.ylabel("F1-score Macro")
plt.title("ViT - F1-score Macro")
plt.legend()
plt.show()

## 15. Avaliação final do Vision Transformer

In [ ]:
vit_test_loss, vit_test_acc, vit_test_f1, y_true_vit, y_pred_vit = evaluate(
    vit_model,
    test_loader,
    criterion
)

print("ViT - Resultados no teste")
print(f"Loss:     {vit_test_loss:.4f}")
print(f"Accuracy: {vit_test_acc:.4f}")
print(f"F1 Macro: {vit_test_f1:.4f}")

## 16. Matrizes de confusão

Como existem 102 classes, uma matriz completa pode ficar visualmente densa. Ainda assim, ela permite identificar padrões de confusão entre espécies visualmente semelhantes.

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, cmap="Blues", cbar=True)
    plt.title(title)
    plt.xlabel("Classe prevista")
    plt.ylabel("Classe real")
    plt.show()

plot_confusion(y_true_cnn, y_pred_cnn, "Matriz de Confusão - CNN ResNet50")
plot_confusion(y_true_vit, y_pred_vit, "Matriz de Confusão - Vision Transformer ViT-B/16")

## 17. Comparação quantitativa entre os modelos

In [ ]:
results = pd.DataFrame([
    {
        "Modelo": "CNN - ResNet50",
        "Accuracy": cnn_test_acc,
        "F1 Macro": cnn_test_f1,
        "Tempo (min)": cnn_time / 60,
        "Parâmetros Totais": cnn_total_params,
        "Parâmetros Treináveis": cnn_trainable_params
    },
    {
        "Modelo": "Vision Transformer - ViT-B/16",
        "Accuracy": vit_test_acc,
        "F1 Macro": vit_test_f1,
        "Tempo (min)": vit_time / 60,
        "Parâmetros Totais": vit_total_params,
        "Parâmetros Treináveis": vit_trainable_params
    }
])

results

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results["Modelo"], results["Accuracy"])
plt.ylabel("Accuracy")
plt.title("Comparação de Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=10)
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(results["Modelo"], results["F1 Macro"])
plt.ylabel("F1-score Macro")
plt.title("Comparação de F1-score Macro")
plt.ylim(0, 1)
plt.xticks(rotation=10)
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(results["Modelo"], results["Tempo (min)"])
plt.ylabel("Minutos")
plt.title("Tempo de Fine-tuning")
plt.xticks(rotation=10)
plt.show()

## 18. Análise de overfitting e underfitting

A interpretação deve ser feita observando as curvas geradas:

- **Overfitting:** a métrica de treino continua melhorando, mas a validação para de melhorar ou piora.
- **Underfitting:** treino e validação permanecem com desempenho baixo.
- **Boa generalização:** treino e validação apresentam resultados próximos e razoavelmente altos.

O código abaixo faz uma análise simples baseada na diferença final entre treino e validação.

In [ ]:
def diagnose_fit(history, model_name):
    train_acc = history["train_acc"][-1]
    val_acc = history["val_acc"][-1]
    gap = train_acc - val_acc

    print(f"Diagnóstico: {model_name}")
    print(f"Accuracy final de treino: {train_acc:.4f}")
    print(f"Accuracy final de validação: {val_acc:.4f}")
    print(f"Gap treino-validação: {gap:.4f}")

    if train_acc < 0.50 and val_acc < 0.50:
        print("Possível underfitting.")
    elif gap > 0.15:
        print("Possível overfitting.")
    else:
        print("Não há um sinal forte de overfitting pelos critérios simples utilizados.")

diagnose_fit(cnn_history, "CNN - ResNet50")
print()
diagnose_fit(vit_history, "Vision Transformer - ViT-B/16")

## 19. Conclusão

A escolha final deve considerar os resultados produzidos nesta execução.

- Se a **ResNet50** apresentar Accuracy e F1 semelhantes ao ViT, ela tende a ser a escolha mais eficiente para um cenário com restrição de hardware, devido à menor complexidade operacional e ao custo computacional normalmente mais previsível.
- Se o **ViT-B/16** apresentar ganho consistente de desempenho e melhor generalização, seu maior custo pode ser justificado, principalmente em uma aplicação de grande escala.
- Como o dataset possui 102 espécies visualmente semelhantes, o pré-treinamento é importante para compensar a quantidade relativamente limitada de imagens de treino por classe.

Para uma empresa de floricultura, a recomendação não deve ser baseada apenas na Accuracy. O modelo deve equilibrar qualidade das classificações, tempo de inferência, custo de infraestrutura e facilidade de manutenção.

**Recomendação final automática:**

In [ ]:
best_model = results.loc[results["F1 Macro"].idxmax(), "Modelo"]

print("Tabela final:")
display(results.sort_values("F1 Macro", ascending=False))

print("\nModelo recomendado com base no maior F1-score macro:")
print(best_model)

print("\nJustificativa:")
if best_model == "CNN - ResNet50":
    print(
        "A CNN apresentou o melhor F1-score macro nesta execução. "
        "Portanto, ela foi a alternativa mais adequada considerando o desempenho observado."
    )
else:
    print(
        "O Vision Transformer apresentou o melhor F1-score macro nesta execução. "
        "Portanto, o ganho de desempenho observado justifica considerar sua maior complexidade."
    )

## 20. Possíveis melhorias futuras

- Realizar fine-tuning progressivo, descongelando mais camadas;
- Utilizar scheduler de learning rate;
- Aplicar early stopping;
- Ajustar data augmentation;
- Testar modelos menores, como ResNet18 e ViT menores;
- Avaliar Top-5 Accuracy;
- Analisar erros entre espécies visualmente semelhantes;
- Realizar validação cruzada quando aplicável;
- Exportar o melhor modelo para uso em produção.